# Notebook 01 — Landing → Raw

**Scope:** full dataset, both taxi types, Jan 2024–May 2026 (29 months each).

This notebook runs the actual Landing → Raw ingestion pipeline implemented in
`src/dataset.py` (the same code `make raw` / `python -m src.dataset all` runs), then
surfaces the resulting row/column counts that back **Table 1** (dataset sizes, TLC row)
and the **Raw** row of **Table 2** (pipeline shape).

This stage standardises column names/types against the canonical schemas in
`src/schemas.py` and attaches provenance columns. It does **not** filter, deduplicate,
or clean anything — every Landing row is preserved as-is (see notebook 02 for the
Raw → Clean rules and their exact removal counts).

## 1. Setup

In [1]:
from pyspark.sql import SparkSession

from src.config import RAW_DATA_DIR
from src.dataset import ingest_taxi_type, write_ingestion_log

spark = (
    SparkSession.builder
    .appName("nyc-taxi-landing-to-raw")
    .master("local[*]")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark version: {spark.version}")

2026-08-28 15:58:53.449 | INFO     | src.config:<module>:26 - PROJ_ROOT path is: /Users/debaoli/Desktop/nyc-taxi-project/nyc_taxi_demand_prediction_examiner_final


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/28 05:58:54 WARN Utils: Your hostname, Debaos-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.2 instead (on interface en0)
26/08/28 05:58:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/08/28 05:58:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.2.0


## 2. Ingest Yellow Taxi (29 months)

Reads every file in `data/landing/yellow/`, standardises to the canonical Yellow Raw
schema, and writes one flat parquet file per month to `data/raw/yellow/`.

In [2]:
yellow_logs = ingest_taxi_type(spark, "yellow")
print(f"Ingested {len(yellow_logs)} Yellow Taxi month-files.")

2026-08-28 05:58:56.496 | INFO     | src.dataset:ingest_file:165 - Reading Yellow Taxi Data Apr 2024.parquet


2026-08-28 05:58:59.909 | INFO     | src.dataset:ingest_file:181 - Writing 3,514,289 rows -> /Users/debaoli/Desktop/nyc-taxi-project/nyc_taxi_demand_prediction_examiner_final/data/raw/yellow/yellow-04-24.parquet


2026-08-28 05:59:05.178 | INFO     | src.dataset:ingest_file:165 - Reading Yellow Taxi Data Apr 2025.parquet


2026-08-28 05:59:06.594 | INFO     | src.dataset:ingest_file:181 - Writing 3,970,553 rows -> /Users/debaoli/Desktop/nyc-taxi-project/nyc_taxi_demand_prediction_examiner_final/data/raw/yellow/yellow-04-25.parquet


2026-08-28 05:59:11.881 | INFO     | src.dataset:ingest_file:165 - Reading Yellow Taxi Data Apr 2026.parquet


...

2026-08-28 06:01:33.718 | INFO     | src.dataset:ingest_file:165 - Reading Yellow Taxi Data Sept 2024.parquet


2026-08-28 06:01:34.374 | INFO     | src.dataset:ingest_file:181 - Writing 3,633,030 rows -> /Users/debaoli/Desktop/nyc-taxi-project/nyc_taxi_demand_prediction_examiner_final/data/raw/yellow/yellow-09-24.parquet


2026-08-28 06:01:39.151 | INFO     | src.dataset:ingest_file:165 - Reading Yellow Taxi Data Sept 2025.parquet


2026-08-28 06:01:39.829 | INFO     | src.dataset:ingest_file:181 - Writing 4,251,015 rows -> /Users/debaoli/Desktop/nyc-taxi-project/nyc_taxi_demand_prediction_examiner_final/data/raw/yellow/yellow-09-25.parquet


Ingested 29 Yellow Taxi month-files.


## 3. Ingest Green Taxi (29 months)

Same process, `data/landing/green/` → `data/raw/green/`.

In [3]:
green_logs = ingest_taxi_type(spark, "green")
print(f"Ingested {len(green_logs)} Green Taxi month-files.")

2026-08-28 06:01:45.399 | INFO     | src.dataset:ingest_file:165 - Reading Green Trip Data Apr 2024.parquet


2026-08-28 06:01:45.793 | INFO     | src.dataset:ingest_file:181 - Writing 56,471 rows -> /Users/debaoli/Desktop/nyc-taxi-project/nyc_taxi_demand_prediction_examiner_final/data/raw/green/green-04-24.parquet
2026-08-28 06:01:45.987 | INFO     | src.dataset:ingest_file:165 - Reading Green Trip Data Apr 2025.parquet


2026-08-28 06:01:46.377 | INFO     | src.dataset:ingest_file:181 - Writing 52,132 rows -> /Users/debaoli/Desktop/nyc-taxi-project/nyc_taxi_demand_prediction_examiner_final/data/raw/green/green-04-25.parquet
2026-08-28 06:01:46.565 | INFO     | src.dataset:ingest_file:165 - Reading Green Trip Data Apr 2026.parquet


2026-08-28 06:01:46.879 | INFO     | src.dataset:ingest_file:181 - Writing 44,238 rows -> /Users/debaoli/Desktop/nyc-taxi-project/nyc_taxi_demand_prediction_examiner_final/data/raw/green/green-04-26.parquet
2026-08-28 06:01:47.058 | INFO     | src.dataset:ingest_file:165 - Reading Green Trip Data Aug 2024.parquet


2026-08-28 06:01:47.369 | INFO     | src.dataset:ingest_file:181 - Writing 51,771 rows -> /Users/debaoli/Desktop/nyc-taxi-project/nyc_taxi_demand_prediction_examiner_final/data/raw/green/green-08-24.parquet
2026-08-28 06:01:47.564 | INFO     | src.dataset:ingest_file:165 - Reading Green Trip Data Aug 2025.parquet


...

2026-08-28 06:01:56.857 | INFO     | src.dataset:ingest_file:181 - Writing 46,912 rows -> /Users/debaoli/Desktop/nyc-taxi-project/nyc_taxi_demand_prediction_examiner_final/data/raw/green/green-11-25.parquet
2026-08-28 06:01:57.032 | INFO     | src.dataset:ingest_file:165 - Reading Green Trip Data Oct 2024.parquet


2026-08-28 06:01:57.289 | INFO     | src.dataset:ingest_file:181 - Writing 56,147 rows -> /Users/debaoli/Desktop/nyc-taxi-project/nyc_taxi_demand_prediction_examiner_final/data/raw/green/green-10-24.parquet
2026-08-28 06:01:57.480 | INFO     | src.dataset:ingest_file:165 - Reading Green Trip Data Oct 2025.parquet


2026-08-28 06:01:57.754 | INFO     | src.dataset:ingest_file:181 - Writing 49,416 rows -> /Users/debaoli/Desktop/nyc-taxi-project/nyc_taxi_demand_prediction_examiner_final/data/raw/green/green-10-25.parquet
2026-08-28 06:01:57.934 | INFO     | src.dataset:ingest_file:165 - Reading Green Trip Data Sept 2024.parquet


2026-08-28 06:01:58.188 | INFO     | src.dataset:ingest_file:181 - Writing 54,440 rows -> /Users/debaoli/Desktop/nyc-taxi-project/nyc_taxi_demand_prediction_examiner_final/data/raw/green/green-09-24.parquet
2026-08-28 06:01:58.372 | INFO     | src.dataset:ingest_file:165 - Reading Green Trip Data Sept 2025.parquet


2026-08-28 06:01:58.639 | INFO     | src.dataset:ingest_file:181 - Writing 48,893 rows -> /Users/debaoli/Desktop/nyc-taxi-project/nyc_taxi_demand_prediction_examiner_final/data/raw/green/green-09-25.parquet
Ingested 29 Green Taxi month-files.


## 4. Ingestion log

Writes the detailed JSON + summary CSV ingestion log to
`reports/data_quality/landing_to_raw_ingestion_{log.json,summary.csv}`, and confirms
every file's row count survived ingestion unchanged (this stage never filters rows).

In [4]:
all_logs = yellow_logs + green_logs
write_ingestion_log(all_logs)

mismatches = [entry for entry in all_logs if entry["status"] != "ok"]
print(f"Ingested {len(all_logs)} file(s) total.")
if mismatches:
    print(f"ROW COUNT MISMATCHES: {len(mismatches)}")
    for m in mismatches:
        print(f"  {m['source_file']}: {m['landing_row_count']:,} -> {m['raw_row_count']:,}")
else:
    print("All row counts preserved Landing -> Raw (no rows filtered at this stage).")

2026-08-28 06:01:58.828 | INFO     | src.dataset:write_ingestion_log:251 - Ingestion log written to /Users/debaoli/Desktop/nyc-taxi-project/nyc_taxi_demand_prediction_examiner_final/reports/data_quality
Ingested 58 file(s) total.
All row counts preserved Landing -> Raw (no rows filtered at this stage).


## 5. Table 1 — dataset size (TLC row)

Row counts summed from the ingestion log; feature count is the observed range of raw
**Landing**-file column counts across all 58 files (varies by month/taxi type — e.g.
`cbd_congestion_fee` only exists in Landing files from 2025 onward, see notebook 00).

In [5]:
yellow_rows = sum(e["landing_row_count"] for e in yellow_logs)
green_rows = sum(e["landing_row_count"] for e in green_logs)
combined_rows = yellow_rows + green_rows

landing_col_counts = [e["landing_column_count"] for e in all_logs]

print("Table 1 — TLC Yellow + Green trips")
print(f"  Yellow rows:        {yellow_rows:,}")
print(f"  Green rows:         {green_rows:,}")
print(f"  Combined rows:      {combined_rows:,}")
print(f"  Feature count range: {min(landing_col_counts)}–{max(landing_col_counts)}")

Table 1 — TLC Yellow + Green trips
  Yellow rows:        108,891,604
  Green rows:         1,462,605
  Combined rows:      110,354,209
  Feature count range: 19–21


## 6. Table 2 — pipeline shape, Raw row

The Raw stage's row counts, confirming Landing = Raw with no rows lost at this stage.

In [6]:
yellow_raw_rows = sum(e["raw_row_count"] for e in yellow_logs)
green_raw_rows = sum(e["raw_row_count"] for e in green_logs)

print("Table 2 — Raw stage")
print(f"  Yellow rows:   {yellow_raw_rows:,}")
print(f"  Green rows:    {green_raw_rows:,}")
print(f"  Combined rows: {yellow_raw_rows + green_raw_rows:,}")
assert yellow_raw_rows == yellow_rows and green_raw_rows == green_rows, (
    "Raw row counts should exactly equal Landing row counts -- this stage never filters."
)
print()
print("Confirmed: Raw row counts exactly match Landing row counts (no rows lost).")


Table 2 — Raw stage
  Yellow rows:   108,891,604
  Green rows:    1,462,605
  Combined rows: 110,354,209

Confirmed: Raw row counts exactly match Landing row counts (no rows lost).


In [7]:
spark.stop()

## Notes

- This notebook only ingests and standardises schema — no filtering, deduplication, or
  cleaning happens here. Raw → Clean row-removal counts are computed in notebook
  `02_raw_to_clean.ipynb`.
- Output: one flat parquet file per month under `data/raw/{yellow,green}/`, plus the
  ingestion log under `reports/data_quality/`.